# Reto 6: Validador de Códigos con Expresiones Regulares

## Programación para Ciencia de Datos
### Instituto Politécnico Nacional — Febrero - Julio 2026

---

In [1]:
import re
import csv
import os
from datetime import date
from typing import Dict, List

# Departamentos válidos para empleados
DEPARTAMENTOS_VALIDOS = ['VEN', 'ADM', 'TEC', 'LOG', 'RHH']

# Series válidas para facturas
SERIES_VALIDAS = ['A', 'B', 'C', 'D', 'E']

## PARTE 1: Funciones de Validación Individual (40%)

In [2]:
def validar_producto(codigo: str) -> Dict:
    """
    Valida código de producto.
    Formato: ABC-1234-MX

    Patrón regex explicado:
      ^           -> inicio de cadena (nada antes)
      ([A-Z]{3})  -> grupo 1: exactamente 3 letras mayúsculas A-Z (categoría)
      -           -> guión literal obligatorio
      (\\d{4})     -> grupo 2: exactamente 4 dígitos 0-9 (número)
      -           -> guión literal obligatorio
      ([A-Z]{2})  -> grupo 3: exactamente 2 letras mayúsculas A-Z (país)
      $           -> fin de cadena (nada después)

    Rechaza: minúsculas, distinto número de caracteres,
             letras donde van dígitos y viceversa.
    """
    resultado = {
        "valido": False,
        "categoria": None,
        "numero": None,
        "pais": None
    }

    patron = r'^([A-Z]{3})-(\d{4})-([A-Z]{2})$'
    match = re.match(patron, codigo)

    if match:
        resultado["valido"]    = True
        resultado["categoria"] = match.group(1)  # ej: 'TEC'
        resultado["numero"]    = match.group(2)  # ej: '0001'
        resultado["pais"]      = match.group(3)  # ej: 'MX'

    return resultado

In [3]:
def validar_envio(codigo: str) -> Dict:
    """
    Valida código de envío.
    Formato: ENV-YYYY-MM-DD-NNNNNN

    Patrón regex explicado:
      ^ENV-                        -> prefijo fijo literal
      (202[0-9]|2030)              -> grupo 1: año 2020-2029 ó exactamente 2030
      -                            -> guión
      (0[1-9]|1[0-2])              -> grupo 2: mes 01-09 ó 10,11,12
      -                            -> guión
      (0[1-9]|[12][0-9]|3[01])     -> grupo 3: día 01-09, 10-29, 30 ó 31
      -                            -> guión
      (\\d{6})                      -> grupo 4: 6 dígitos secuencial
      $                            -> fin de cadena

    Nota: valida rangos numéricos dentro del regex, no solo formato.
    """
    resultado = {
        "valido": False,
        "fecha": None,
        "secuencial": None
    }

    patron = r'^ENV-(202[0-9]|2030)-(0[1-9]|1[0-2])-(0[1-9]|[12][0-9]|3[01])-(\d{6})$'
    match = re.match(patron, codigo)

    if match:
        resultado["valido"]     = True
        resultado["fecha"]      = f"{match.group(1)}-{match.group(2)}-{match.group(3)}"
        resultado["secuencial"] = match.group(4)

    return resultado

In [4]:
def validar_empleado(codigo: str) -> Dict:
    """
    Valida código de empleado.
    Formato: EMP-XXX-NNNN

    Patrón regex explicado:
      ^EMP-          -> prefijo fijo literal
      ([A-Z]{3})     -> grupo 1: 3 letras mayúsculas (departamento)
      -              -> guión
      ([1-9]\\d{3})   -> grupo 2: dígito 1-9 seguido de 3 dígitos
                        esto garantiza que NO empiece con 0
                        rango efectivo: 1000-9999
      $              -> fin de cadena

    Validación extra: el departamento debe estar en DEPARTAMENTOS_VALIDOS.
    El regex solo valida el formato; la lista verifica el valor.
    """
    resultado = {
        "valido": False,
        "departamento": None,
        "numero": None
    }

    patron = r'^EMP-([A-Z]{3})-([1-9]\d{3})$'
    match = re.match(patron, codigo)

    if match:
        depto = match.group(1)
        # Verificación semántica: el depto debe existir en la empresa
        if depto in DEPARTAMENTOS_VALIDOS:
            resultado["valido"]       = True
            resultado["departamento"] = depto
            resultado["numero"]       = match.group(2)

    return resultado

In [5]:
def validar_factura(codigo: str) -> Dict:
    """
    Valida código de factura.
    Formato: FAC-S-NNNNNN

    Patrón regex explicado:
      ^FAC-          -> prefijo fijo literal
      ([A-E])        -> grupo 1: UNA letra mayúscula entre A y E
                        [A-E] es un rango de caracteres: A,B,C,D,E
                        rechaza F,G,...,Z y cualquier minúscula
      -              -> guión
      (\\d{6})        -> grupo 2: exactamente 6 dígitos
      $              -> fin de cadena
    """
    resultado = {
        "valido": False,
        "serie": None,
        "numero": None
    }

    patron = r'^FAC-([A-E])-(\d{6})$'
    match = re.match(patron, codigo)

    if match:
        resultado["valido"]  = True
        resultado["serie"]   = match.group(1)
        resultado["numero"]  = match.group(2)

    return resultado

In [6]:
# Prueba de todas las funciones individuales
print('PRUEBA DE FUNCIONES INDIVIDUALES')
print('=' * 50)

print('\n-- Productos --')
for c in ['TEC-0001-MX','ALI-9999-US','ROB-1234-CA','tec-0001-MX','TEC-001-MX','TECH-0001-MX']:
    print(validar_producto(c))

print('\n-- Envíos --')
for c in ['ENV-2024-03-15-001234','ENV-2025-12-01-999999',
          'ENV-2019-03-15-001234','ENV-2024-13-15-001234','ENV-2024-03-32-001234']:
    print(validar_envio(c))

print('\n-- Empleados --')
for c in ['EMP-VEN-1234','EMP-TEC-9999','EMP-ADM-1000',
          'EMP-VEN-0123','EMP-XXX-1234','EMP-VEN-123']:
    print(validar_empleado(c))

print('\n-- Facturas --')
for c in ['FAC-A-123456','FAC-E-000001','FAC-B-999999',
          'FAC-F-123456','FAC-A-12345','FAC-a-123456']:
    print(validar_factura(c))

PRUEBA DE FUNCIONES INDIVIDUALES

-- Productos --
{'valido': True, 'categoria': 'TEC', 'numero': '0001', 'pais': 'MX'}
{'valido': True, 'categoria': 'ALI', 'numero': '9999', 'pais': 'US'}
{'valido': True, 'categoria': 'ROB', 'numero': '1234', 'pais': 'CA'}
{'valido': False, 'categoria': None, 'numero': None, 'pais': None}
{'valido': False, 'categoria': None, 'numero': None, 'pais': None}
{'valido': False, 'categoria': None, 'numero': None, 'pais': None}

-- Envíos --
{'valido': True, 'fecha': '2024-03-15', 'secuencial': '001234'}
{'valido': True, 'fecha': '2025-12-01', 'secuencial': '999999'}
{'valido': False, 'fecha': None, 'secuencial': None}
{'valido': False, 'fecha': None, 'secuencial': None}
{'valido': False, 'fecha': None, 'secuencial': None}

-- Empleados --
{'valido': True, 'departamento': 'VEN', 'numero': '1234'}
{'valido': True, 'departamento': 'TEC', 'numero': '9999'}
{'valido': True, 'departamento': 'ADM', 'numero': '1000'}
{'valido': False, 'departamento': None, 'numero': 

## PARTE 2: Validador Universal (30%)

In [7]:
def validar_codigo(codigo: str) -> Dict:
    """
    Detecta automáticamente el tipo de código por su prefijo
    y delega la validación a la función especializada.

    Estrategia de detección:
      - Empieza con 'ENV-'       → envio
      - Empieza con 'EMP-'       → empleado
      - Empieza con 'FAC-'       → factura
      - Empieza con 3 mayúsculas → producto (puede ser inválido igualmente)
      - Cualquier otro caso      → desconocido

    El regex r'^[A-Z]{3}-' detecta posibles productos sin
    confundirlos con ENV/EMP/FAC (esos se evalúan primero).
    """
    resultado = {
        "codigo": codigo,
        "tipo": "desconocido",
        "valido": False,
        "detalles": {}
    }

    if codigo.startswith('ENV-'):
        resultado['tipo'] = 'envio'
        detalles = validar_envio(codigo)
    elif codigo.startswith('EMP-'):
        resultado['tipo'] = 'empleado'
        detalles = validar_empleado(codigo)
    elif codigo.startswith('FAC-'):
        resultado['tipo'] = 'factura'
        detalles = validar_factura(codigo)
    elif re.match(r'^[A-Z]{3}-', codigo):
        resultado['tipo'] = 'producto'
        detalles = validar_producto(codigo)
    else:
        return resultado  # desconocido, sin detalles

    resultado['valido']   = detalles.pop('valido')
    resultado['detalles'] = detalles
    return resultado


def mostrar_resultado(resultado: Dict) -> None:
    """Muestra el resultado de validación de forma legible."""
    estado = '✓' if resultado['valido'] else '✗'
    print(f"{estado} {resultado['codigo']:<30} | Tipo: {resultado['tipo']:<12}")
    if resultado['valido'] and resultado['detalles']:
        detalles = ', '.join(f"{k}: {v}" for k, v in resultado['detalles'].items() if v)
        print(f'   └── {detalles}')

In [8]:
CODIGOS_PRUEBA = [
    # Productos
    'TEC-0001-MX', 'ALI-9999-US', 'ROB-1234-CA',
    'tec-0001-MX', 'TEC-001-MX',  'TECH-0001-MX',
    # Envíos
    'ENV-2024-03-15-001234', 'ENV-2025-12-01-999999',
    'ENV-2019-03-15-001234', 'ENV-2024-13-15-001234', 'ENV-2024-03-32-001234',
    # Empleados
    'EMP-VEN-1234', 'EMP-TEC-9999', 'EMP-ADM-1000',
    'EMP-VEN-0123', 'EMP-XXX-1234', 'EMP-VEN-123',
    # Facturas
    'FAC-A-123456', 'FAC-E-000001', 'FAC-B-999999',
    'FAC-F-123456', 'FAC-A-12345',  'FAC-a-123456',
    # Desconocidos
    'XXX-1234', 'RANDOM-CODE',
]

print('PRUEBA DE VALIDADOR UNIVERSAL')
print('=' * 50)
for codigo in CODIGOS_PRUEBA:
    mostrar_resultado(validar_codigo(codigo))

PRUEBA DE VALIDADOR UNIVERSAL
✓ TEC-0001-MX                    | Tipo: producto    
   └── categoria: TEC, numero: 0001, pais: MX
✓ ALI-9999-US                    | Tipo: producto    
   └── categoria: ALI, numero: 9999, pais: US
✓ ROB-1234-CA                    | Tipo: producto    
   └── categoria: ROB, numero: 1234, pais: CA
✗ tec-0001-MX                    | Tipo: desconocido 
✗ TEC-001-MX                     | Tipo: producto    
✗ TECH-0001-MX                   | Tipo: desconocido 
✓ ENV-2024-03-15-001234          | Tipo: envio       
   └── fecha: 2024-03-15, secuencial: 001234
✓ ENV-2025-12-01-999999          | Tipo: envio       
   └── fecha: 2025-12-01, secuencial: 999999
✗ ENV-2019-03-15-001234          | Tipo: envio       
✗ ENV-2024-13-15-001234          | Tipo: envio       
✗ ENV-2024-03-32-001234          | Tipo: envio       
✓ EMP-VEN-1234                   | Tipo: empleado    
   └── departamento: VEN, numero: 1234
✓ EMP-TEC-9999                   | Tipo: empleado    
 

## PARTE 3: Procesamiento por Lotes (30%)

In [9]:
def procesar_lote(codigos: List[str]) -> Dict:
    """
    Procesa una lista de códigos y genera estadísticas.
    Para cada código llama a validar_codigo() y acumula
    contadores globales y por tipo.
    """
    resultado = {
        'total': 0,
        'validos': 0,
        'invalidos': 0,
        'por_tipo': {
            'producto':    {'total': 0, 'validos': 0},
            'envio':       {'total': 0, 'validos': 0},
            'empleado':    {'total': 0, 'validos': 0},
            'factura':     {'total': 0, 'validos': 0},
            'desconocido': {'total': 0, 'validos': 0}
        },
        'detalle': []
    }

    for codigo in codigos:
        res = validar_codigo(codigo)
        resultado['total'] += 1

        if res['valido']:
            resultado['validos'] += 1
        else:
            resultado['invalidos'] += 1

        tipo = res['tipo']
        resultado['por_tipo'][tipo]['total'] += 1
        if res['valido']:
            resultado['por_tipo'][tipo]['validos'] += 1

        resultado['detalle'].append(res)

    return resultado


def mostrar_reporte(reporte: Dict) -> None:
    """Muestra el reporte de procesamiento por lotes."""
    print('=' * 60)
    print('                 REPORTE DE VALIDACIÓN')
    print('=' * 60)
    print(f"\nTotal procesados: {reporte['total']}")
    print(f"Válidos:   {reporte['validos']} ({reporte['validos']/reporte['total']*100:.1f}%)")
    print(f"Inválidos: {reporte['invalidos']} ({reporte['invalidos']/reporte['total']*100:.1f}%)")
    print('\nDesglose por tipo:')
    print('-' * 40)
    for tipo, stats in reporte['por_tipo'].items():
        if stats['total'] > 0:
            tasa = stats['validos'] / stats['total'] * 100
            print(f"  {tipo.capitalize():<12}: {stats['validos']:>3}/{stats['total']:<3} ({tasa:.0f}% válidos)")
    print('\n' + '=' * 60)

In [10]:
reporte = procesar_lote(CODIGOS_PRUEBA)
mostrar_reporte(reporte)

                 REPORTE DE VALIDACIÓN

Total procesados: 25
Válidos:   11 (44.0%)
Inválidos: 14 (56.0%)

Desglose por tipo:
----------------------------------------
  Producto    :   3/5   (60% válidos)
  Envio       :   2/5   (40% válidos)
  Empleado    :   3/6   (50% válidos)
  Factura     :   3/6   (50% válidos)
  Desconocido :   0/3   (0% válidos)



## BONUS: Funcionalidades Extra (+10)

In [11]:
# ── BONUS 1: Sugerencia de corrección ────────────────────────────────────
def sugerir_correccion(codigo: str) -> str:
    """
    Intenta corregir un código inválido automáticamente.
    Estrategia: convertir a mayúsculas y eliminar espacios.
    Si la versión corregida es válida, la sugiere.
    """
    sugerencia = codigo.strip().upper()
    if not validar_codigo(codigo)['valido'] and validar_codigo(sugerencia)['valido']:
        return f"¿Quisiste decir: '{sugerencia}'?"
    elif not validar_codigo(codigo)['valido']:
        return 'No se encontró corrección automática.'
    return 'El código ya es válido.'


# ── BONUS 2: Validación de fecha real ────────────────────────────────────
def validar_fecha_real(anio: int, mes: int, dia: int) -> bool:
    """
    Valida que la fecha exista en el calendario real usando datetime.
    Detecta casos como: 31 de febrero, 31 de abril, año no bisiesto.
    """
    try:
        date(anio, mes, dia)
        return True
    except ValueError:
        return False


def validar_envio_estricto(codigo: str) -> Dict:
    """
    Versión mejorada: valida formato (regex) Y fecha real (datetime).
    """
    resultado = validar_envio(codigo)
    if resultado['valido']:
        anio, mes, dia = map(int, resultado['fecha'].split('-'))
        if not validar_fecha_real(anio, mes, dia):
            resultado.update({'valido': False, 'fecha': None, 'secuencial': None})
    return resultado


# ── BONUS 3: Exportar a CSV ───────────────────────────────────────────────
def exportar_resultados(reporte: Dict, archivo: str) -> None:
    """
    Guarda el detalle de validación en un archivo CSV.
    Columnas: codigo, tipo, valido, detalles.
    """
    os.makedirs(os.path.dirname(archivo) or '.', exist_ok=True)
    with open(archivo, 'w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow(['codigo', 'tipo', 'valido', 'detalles'])
        for item in reporte['detalle']:
            detalles_str = ' | '.join(
                f"{k}={v}" for k, v in item['detalles'].items() if v
            )
            writer.writerow([item['codigo'], item['tipo'], item['valido'], detalles_str])
    print(f'Resultados exportados a: {archivo}')

In [12]:
print('BONUS 1 — Sugerencia de corrección')
print('-' * 40)
for c in ['tec-0001-mx', 'fac-a-123456', 'TEC-001-MX', 'TEC-0001-MX']:
    print(f"  sugerir_correccion('{c}') → {sugerir_correccion(c)}")

print('\nBONUS 2 — Validación de fecha real')
print('-' * 40)
for a, m, d in [(2024,2,29),(2023,2,29),(2024,4,31),(2024,3,15)]:
    print(f'  validar_fecha_real({a},{m:02d},{d:02d}) → {validar_fecha_real(a,m,d)}')
print()
for c in ['ENV-2024-02-29-001234','ENV-2023-02-29-001234','ENV-2024-04-31-001234']:
    print(f"  validar_envio_estricto('{c}') → {validar_envio_estricto(c)}")

print('\nBONUS 3 — Exportar a CSV')
print('-' * 40)
exportar_resultados(reporte, 'outputs/resultados_validacion.csv')
with open('outputs/resultados_validacion.csv') as f:
    lines = f.readlines()
for l in lines[:6]:
    print(l, end='')
if len(lines) > 6:
    print(f'... ({len(lines)-1} filas en total)')

BONUS 1 — Sugerencia de corrección
----------------------------------------
  sugerir_correccion('tec-0001-mx') → ¿Quisiste decir: 'TEC-0001-MX'?
  sugerir_correccion('fac-a-123456') → ¿Quisiste decir: 'FAC-A-123456'?
  sugerir_correccion('TEC-001-MX') → No se encontró corrección automática.
  sugerir_correccion('TEC-0001-MX') → El código ya es válido.

BONUS 2 — Validación de fecha real
----------------------------------------
  validar_fecha_real(2024,02,29) → True
  validar_fecha_real(2023,02,29) → False
  validar_fecha_real(2024,04,31) → False
  validar_fecha_real(2024,03,15) → True

  validar_envio_estricto('ENV-2024-02-29-001234') → {'valido': True, 'fecha': '2024-02-29', 'secuencial': '001234'}
  validar_envio_estricto('ENV-2023-02-29-001234') → {'valido': False, 'fecha': None, 'secuencial': None}
  validar_envio_estricto('ENV-2024-04-31-001234') → {'valido': False, 'fecha': None, 'secuencial': None}

BONUS 3 — Exportar a CSV
----------------------------------------
Resultados e